# Yelp 증강데이터 Hand-craft Feature 추출

`Yelp/yelp_dataset/data_yelp.parquet` (20,000행, Human-vs-AI 라벨링)의 `text` 컬럼에서
언어학적 hand-craft feature를 추출해 원본 컬럼 옆에 붙여 저장한다.

| 그룹 | 출력 컬럼 | 라이브러리 |
|---|---|---|
| Counts | `syllable` `lexicon` `sentence` `char` `letter` `polysyllab` `monosyllab` | textstat |
| POS | `nouns` `adj` `verbs` `pronoun` `adverb` `article` | spaCy `en_core_web_sm` |
| Readability | `smog_index` `flesch_reading_ease` `flesch_kincaid_grade` `fog_scale` `dale_chall` `reading_time` | textstat |
| Sentiment | `sentiment` `subjectivity` | TextBlob |
| LM | `perplexity` `burstiness` | GPT-2 (transformers) |

- **입력**: `Yelp/yelp_dataset/data_yelp.parquet`
- **출력**: `Yelp/yelp_dataset/data_yelp_features.parquet` (원본 7컬럼 + 23 feature = 30컬럼)
- perplexity가 GPT-2라 20K 전체는 시간이 걸린다 → 100건마다 체크포인트 저장.

> 커널은 `analysis/.venv` 를 사용한다 (torch/spacy/transformers/textstat 이미 설치됨).

### 1. 추가 의존성 설치
`pyarrow`(parquet), `textblob` 을 현재 커널에 설치. (한 번만 실행하면 됨)

In [1]:
# analysis/.venv 는 uv로 관리되어 pip이 없을 수 있음 → uv로 설치 (이미 있으면 no-op).
# pip 커널이면 대신: %pip install -q pyarrow textblob
import sys, subprocess
subprocess.run(["uv", "pip", "install", "--python", sys.executable,
                "pyarrow", "textblob"], check=False)

Using Python 3.11.15 environment at: /Users/user/project/analysis/.venv
Checked 2 packages in 26ms


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/user/project/analysis/.venv/bin/python', 'pyarrow', 'textblob'], returncode=0)

### 2. 임포트 및 모델 초기화
GPT-2 / spaCy 로드는 시간이 걸리므로 한 번만 실행한다.

In [2]:
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import textstat
import spacy
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from textblob import TextBlob

# --- spaCy (POS만 필요 → ner/parser 비활성) ---
NLP = spacy.load("en_core_web_sm", disable=["ner", "parser"])

# --- GPT-2 (perplexity용) ---
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)
GPT2_TOK = GPT2TokenizerFast.from_pretrained("gpt2")
GPT2_MODEL = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
GPT2_MODEL.eval()

ARTICLES = {"a", "an", "the"}
print("setup done.")

device: mps


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11816.29it/s]


setup done.


### 3. 입력 데이터 로드

In [3]:
INPUT = "../data/data_yelp.parquet"
OUT   = "../data/data_yelp_features.parquet"

df = pd.read_parquet(INPUT)
print("shape:", df.shape)
print("columns:", list(df.columns))
df.head(3)

shape: (20000, 7)
columns: ['pk', 'review_id', 'text', 'label', 'source', 'review_stars', 'business_id']


,pk,review_id,text,label,source,review_stars,business_id
0,1OAdtkX02Omi8qDiycNjjQ_human,1OAdtkX02Omi8qDiycNjjQ,"I was so excited to visit, however the excitem...",human,human,1,ysvzP0VvMTrGmUClKJ3img
1,HIiyQb2na4_vzSzJVa1hVA_human,HIiyQb2na4_vzSzJVa1hVA,The service was horrible. The girl that was te...,human,human,1,qONVcsU_vo3KFve4PtZqpg
2,BXl5P7AloWFJvFKrN7-xcQ_human,BXl5P7AloWFJvFKrN7-xcQ,Cheap beer and outside seating. Essentials to ...,human,human,3,I4Szupt_YHzR9dczcNzfeA


### 4. textstat 카운트 feature
`syllable, lexicon, sentence, char, letter, polysyllab, monosyllab`

In [4]:
_COUNT_KEYS = ["syllable", "lexicon", "sentence", "char", "letter", "polysyllab", "monosyllab"]

def textstat_counts(text: str) -> dict:
    if not isinstance(text, str) or not text.strip():
        return dict.fromkeys(_COUNT_KEYS, 0)
    return {
        "syllable":   textstat.syllable_count(text),
        "lexicon":    textstat.lexicon_count(text),
        "sentence":   textstat.sentence_count(text),
        "char":       textstat.char_count(text),
        "letter":     textstat.letter_count(text),
        "polysyllab": textstat.polysyllabcount(text),
        "monosyllab": textstat.monosyllabcount(text),
    }

### 5. spaCy POS 카운트
`nouns, adj, verbs, pronoun, adverb, article` — 미리 계산한 spaCy doc을 재사용.

In [5]:
_POS_KEYS = ["nouns", "adj", "verbs", "pronoun", "adverb", "article"]

def pos_counts(doc) -> dict:
    if doc is None or len(doc) == 0:
        return dict.fromkeys(_POS_KEYS, 0)
    c = {"nouns": 0, "adj": 0, "verbs": 0, "pronoun": 0, "adverb": 0, "article": 0}
    for tok in doc:
        p = tok.pos_
        if p == "NOUN":   c["nouns"] += 1
        elif p == "ADJ":  c["adj"] += 1
        elif p == "VERB": c["verbs"] += 1
        elif p == "PRON": c["pronoun"] += 1
        elif p == "ADV":  c["adverb"] += 1
        if tok.lower_ in ARTICLES:  # LIWC article = a/an/the
            c["article"] += 1
    return c

### 6. Readability
`smog_index, flesch_reading_ease, flesch_kincaid_grade, fog_scale, dale_chall, reading_time`

In [6]:
_READ_KEYS = ["smog_index", "flesch_reading_ease", "flesch_kincaid_grade",
              "fog_scale", "dale_chall", "reading_time"]

def readability(text: str) -> dict:
    try:
        return {
            "smog_index":           textstat.smog_index(text),
            "flesch_reading_ease":  textstat.flesch_reading_ease(text),
            "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
            "fog_scale":            textstat.gunning_fog(text),
            "dale_chall":           textstat.dale_chall_readability_score(text),
            "reading_time":         textstat.reading_time(text, ms_per_char=14.69),
        }
    except Exception:
        return dict.fromkeys(_READ_KEYS, 0.0)

### 7. Sentiment (TextBlob)
`sentiment` = polarity, `subjectivity` = subjectivity

In [7]:
def sentiment_feats(text: str) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {"sentiment": 0.0, "subjectivity": 0.0}
    s = TextBlob(text).sentiment
    return {"sentiment": s.polarity, "subjectivity": s.subjectivity}

### 9. Perplexity + Burstiness (GPT-2)
문장 단위 PPL의 평균 = `perplexity`, 변동계수(std/mean) = `burstiness`.
(`analysis/extract_features.py:perplexity_features` 로직 축약)

In [8]:
@torch.inference_mode()
def perplexity_feats(text: str, max_length: int = 512) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {"perplexity": 0.0, "burstiness": 0.0}
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if len(s.split()) >= 3]
    if not sents:
        sents = [text]
    ppls = []
    for s in sents:
        try:
            ids = GPT2_TOK(s, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(DEVICE)
            if ids.shape[1] < 2:
                continue
            loss = GPT2_MODEL(ids, labels=ids).loss
            ppls.append(float(torch.exp(loss).item()))
        except Exception:
            continue
    if not ppls:
        return {"perplexity": 0.0, "burstiness": 0.0}
    arr = np.array(ppls, dtype=float)
    mean = float(arr.mean())
    return {"perplexity": mean, "burstiness": float(arr.std() / mean) if mean > 0 else 0.0}

### 10. 단일 텍스트 통합 함수 + 스모크 테스트
spaCy doc은 한 번만 계산해 POS에 재사용.

In [9]:
def extract_all(text: str) -> dict:
    feats = {}
    feats.update(textstat_counts(text))
    feats.update(readability(text))
    feats.update(sentiment_feats(text))
    feats.update(perplexity_feats(text))
    doc = NLP(text[:100_000]) if isinstance(text, str) and text.strip() else None
    feats.update(pos_counts(doc))
    return feats

# 스모크 테스트: 첫 행에 23개 feature가 모두 채워지는지 확인
_sample = extract_all(df["text"].iloc[0])
print("feature 수:", len(_sample))
_sample

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


feature 수: 23


{'syllable': 225,
 'lexicon': 169,
 'sentence': 7,
 'char': 730,
 'letter': 714,
 'polysyllab': 8,
 'monosyllab': 126,
 'smog_index': 9.236282656511921,
 'flesch_reading_ease': 69.69686390532547,
 'flesch_kincaid_grade': 9.535773457311919,
 'fog_scale': 10.840574809805581,
 'dale_chall': 8.384399915469146,
 'reading_time': 10.7237,
 'sentiment': -0.014259259259259248,
 'subjectivity': 0.5092592592592593,
 'perplexity': 136.89585800170897,
 'burstiness': 0.572738368757545,
 'nouns': 35,
 'adj': 9,
 'verbs': 26,
 'pronoun': 19,
 'adverb': 11,
 'article': 13}

### 11. 전체 20K 추출 (100건마다 체크포인트 저장)

In [ ]:
LIMIT = None
CHECKPOINT_EVERY = 100

work = df if LIMIT is None else df.head(LIMIT).copy()
texts = work["text"].fillna("").astype(str).tolist()

rows = []
for i, txt in enumerate(tqdm(texts, total=len(texts))):
    rows.append(extract_all(txt))
    if (i + 1) % CHECKPOINT_EVERY == 0:
        partial = pd.concat(
            [work.iloc[:i + 1].reset_index(drop=True), pd.DataFrame(rows)], axis=1
        )
        partial.to_parquet(OUT, index=False)

out_df = pd.concat([work.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
out_df.to_parquet(OUT, index=False)
print("saved:", OUT, "shape:", out_df.shape)

100%|██████████| 20000/20000 [21:48<00:00, 15.29it/s]

saved: /Users/user/project/Yelp/yelp_dataset/data_yelp_features.parquet shape: (20000, 30)


### 12. 결과 확인

In [11]:
res = pd.read_parquet(OUT)
print("shape:", res.shape)
feature_cols = [c for c in res.columns if c not in df.columns]
print("추출된 feature(", len(feature_cols), "개):", feature_cols)
res[feature_cols].describe().T

shape: (20000, 30)
추출된 feature( 23 개): ['syllable', 'lexicon', 'sentence', 'char', 'letter', 'polysyllab', 'monosyllab', 'smog_index', 'flesch_reading_ease', 'flesch_kincaid_grade', 'fog_scale', 'dale_chall', 'reading_time', 'sentiment', 'subjectivity', 'perplexity', 'burstiness', 'nouns', 'adj', 'verbs', 'pronoun', 'adverb', 'article']


,count,mean,std,min,25%,50%,75%,max
syllable,20000.0,128.520900,73.146041,2.000000,74.000000,112.000000,165.000000,457.000000
lexicon,20000.0,88.523850,52.194243,2.000000,50.000000,76.000000,113.000000,300.000000
sentence,20000.0,6.551150,3.684476,1.000000,4.000000,6.000000,8.000000,38.000000
char,20000.0,408.411350,231.315084,8.000000,237.000000,356.000000,524.000000,1439.000000
letter,20000.0,393.162400,222.871420,7.000000,228.000000,343.000000,505.250000,1388.000000
polysyllab,20000.0,9.175000,5.904415,0.000000,5.000000,8.000000,13.000000,45.000000
monosyllab,20000.0,61.037650,37.902279,0.000000,34.000000,51.000000,77.000000,238.000000
smog_index,20000.0,9.835338,2.093597,3.129100,8.418075,9.827889,11.208143,22.918634
flesch_reading_ease,20000.0,68.781576,12.826165,-133.595000,60.589643,69.760121,77.929596,120.205000
flesch_kincaid_grade,20000.0,7.103162,2.459899,-3.010000,5.460108,6.995750,8.612000,44.709009
